In [0]:
from pyspark.sql import functions as F

import mlflow
import mlflow.sklearn

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


In [0]:
events = spark.table("day8_catalog.ecommerce.events_silver")

ml_df = (
    events
    .groupBy("user_id")
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event_type") == "cart", 1).otherwise(0)).alias("cart_adds"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases")
    )
)

pdf = ml_df.toPandas()

X = pdf[["views", "cart_adds"]]
y = pdf["purchases"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [0]:
results = [
    ("linear_regression", 0.42),   # replace with your actual scores
    ("decision_tree", 0.55),
    ("random_forest", 0.61)
]

best_model = max(results, key=lambda x: x[1])
print(f"Best Model: {best_model[0]} with R² = {best_model[1]:.4f}")


In [0]:
best_model = max(results, key=lambda x: x[1])
print(f"Best Model: {best_model[0]} with R² = {best_model[1]:.4f}")


In [0]:
from sklearn.ensemble import RandomForestRegressor

# Train Random Forest again
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


In [0]:
feature_importance = list(zip(X.columns, rf_model.feature_importances_))
feature_importance


In [0]:
spark_ml_df = ml_df.select("views", "cart_adds", "purchases")


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression as SparkLR

assembler = VectorAssembler(
    inputCols=["views", "cart_adds"],
    outputCol="features"
)

lr = SparkLR(
    featuresCol="features",
    labelCol="purchases"
)

pipeline = Pipeline(stages=[assembler, lr])


In [0]:
train_df, test_df = spark_ml_df.randomSplit([0.8, 0.2], seed=42)

spark_model = pipeline.fit(train_df)

predictions = spark_model.transform(test_df)

predictions.select("features", "purchases", "prediction").show(5)
